# Analise Climática do Oceano Pacífico
Analise estatística e gráfica do aumento da temperatura dos oceanos e a intencificação do fenômeno El Niño.

In [ ]:
#Instalação do arquivo com os dados
!wget -O sst_data.nc "https://downloads.psl.noaa.gov/Datasets/noaa.ersst.v5/sst.mnmean.nc"

#Instação das Bibliotecas
!pip install xarray netCDF4 matplotlib cartopy pandas numpy
!apt-get install -y libgeos-dev

--2026-09-22 18:12:30--  https://downloads.psl.noaa.gov/Datasets/noaa.ersst.v5/sst.mnmean.nc
Resolving downloads.psl.noaa.gov (downloads.psl.noaa.gov)... 140.172.38.87
Connecting to downloads.psl.noaa.gov (downloads.psl.noaa.gov)|140.172.38.87|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 159094577 (152M) [application/x-netcdf]
Saving to: ‘sst_data.nc’

sst_data.nc           0%[                    ]  79.51K   278KB/s               

# Evoluçao das Anomalias de Temperatura e Tendência de aquecimento
Analisa todos os dados de temperatura da série histórica, verificando as anomalias frias, quentes e a tendência de aquecimendo (de acordo com os dados dos ultimos 3 anos)

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from scipy.stats import linregress
import xarray as xr

# Carregar e ajustar coordenadas
ds = xr.open_dataset("sst_data.nc")
sst = ds["sst"].squeeze()

if sst.lon.max() > 180:
    sst = sst.assign_coords(lon=(((sst.lon + 180) % 360) - 180)).sortby("lon")

# Recortar a Costa do Pacífico Sul-Americano
lat_min, lat_max = -10, 0
lon_min, lon_max = -90, -80

sst_region = sst.sel(
    lat=slice(lat_max, lat_min) if sst.lat[0] > sst.lat[-1] else slice(lat_min, lat_max),
    lon=slice(lon_min, lon_max),
)

# Calcular Média Espacial Ponderada
weights = np.cos(np.deg2rad(sst_region.lat))
sst_series = sst_region.weighted(weights).mean(dim=["lat", "lon"])

# Calcular a Anomalia (Linha de base: 1991-2020)
climatology = (
    sst_series.sel(time=slice("1991-01-01", "2020-12-31"))
    .groupby("time.month")
    .mean("time")
)
ssta_series = (sst_series.groupby("time.month") - climatology).to_series()

# Estrutura do DataFrame e Tendências
df = pd.DataFrame({"anomalia": ssta_series})
df["time"] = df.index
df["tendencia"] = (
    df["anomalia"].rolling(window=36, center=True, min_periods=12).mean()
)

# Regressão Linear Secular para evidenciar a inclinação do crescimento
x_years = (df["time"] - df["time"].iloc[0]).dt.days / 365.25
slope, intercept, _, _, _ = linregress(x_years, df["anomalia"])
df["regressao"] = intercept + slope * x_years

# Separação para coloração de anomalias
df["pos"] = df["anomalia"].clip(lower=0)
df["neg"] = df["anomalia"].clip(upper=0)

# Arredondamento para garantir 2 casas decimais no hover unificado
df["anomalia"] = df["anomalia"].round(2)
df["tendencia"] = df["tendencia"].round(2)
df["regressao"] = df["regressao"].round(2)

# --- CONSTRUÇÃO DO GRÁFICO ---
fig = go.Figure()

# Áreas de Anomalia Positiva (Vermelho)
fig.add_trace(
    go.Scatter(
        x=df["time"],
        y=df["pos"],
        mode="lines",
        line=dict(width=0),
        fill="tozeroy",
        fillcolor="rgba(239, 85, 59, 0.35)",
        name="Anomalia Quente (°C)",
        hoverinfo="skip",
    )
)

# Áreas de Anomalia Negativa (Azul)
fig.add_trace(
    go.Scatter(
        x=df["time"],
        y=df["neg"],
        mode="lines",
        line=dict(width=0),
        fill="tozeroy",
        fillcolor="rgba(99, 110, 250, 0.35)",
        name="Anomalia Fria (°C)",
        hoverinfo="skip",
    )
)

# Anomalia Mensal Bruta
fig.add_trace(
    go.Scatter(
        x=df["time"],
        y=df["anomalia"],
        mode="lines",
        line=dict(color="rgba(180, 180, 180, 0.3)", width=0.6),
        name="Anomalia Mensal",
        hovertemplate="%{x|%b %Y}: <b>%{y:+.2f} °C</b><extra></extra>",
    )
)

# Tendência de Médio Prazo (3 Anos)
fig.add_trace(
    go.Scatter(
        x=df["time"],
        y=df["tendencia"],
        mode="lines",
        line=dict(color="#FFD700", width=2.5),
        name="Média Móvel (3 Anos)",
        hovertemplate="Média 3A: <b>%{y:+.2f} °C</b><extra></extra>",
    )
)

# Linha de Crescimento Secular (Regressão Linear de Longo Prazo)
fig.add_trace(
    go.Scatter(
        x=df["time"],
        y=df["regressao"],
        mode="lines",
        line=dict(color="#FF3333", width=3, dash="dash"),
        name=f"Tendência Linear (+{slope*10:.2f}°C/década)",
        hovertemplate="Tendência Linear: <b>%{y:+.2f} °C</b><extra></extra>",
    )
)

# Layout com Aumento do Zoom (Afastamento) e Divisão Y = 0.5
fig.update_layout(
    title="<b>Anomalias de Temperatura e Tendência de Aquecimento</b><br><sup>Costa Pacífica da América do Sul (1860–2026) — Linha de Base Climatológica: 1991–2020</sup>",
    xaxis=dict(
        title="Ano",
        type="date",
        showgrid=True,
        gridcolor="rgba(255, 255, 255, 0.1)",
        # Afastamento lateral estendendo o range de datas nas extremidades
        range=[
            df["time"].min() - pd.Timedelta(days=1825),  #5 anos de margem à esquerda
            df["time"].max() + pd.Timedelta(days=1825),  #5 anos de margem à direita
        ],
    ),
    yaxis=dict(
        title="Desvio de Temperatura (°C em relação à média)",
        dtick=0.5,  # Divisão em blocos de 0.5°C
        gridcolor="rgba(255, 255, 255, 0.12)",
        zeroline=True,
        zerolinewidth=2,
        zerolinecolor="rgba(255, 255, 255, 0.6)",
        # Expansão dos limites Y para afastar o gráfico do topo e da base
        range=[df["anomalia"].min() - 0.8, df["anomalia"].max() + 0.8],
        hoverformat="+.2f",
    ),
    template="plotly_dark",
    hovermode="x unified",
    height=600,
    margin=dict(r=30, t=90, l=30, b=110),
    legend=dict(
        orientation="h",
        yanchor="top",
        y=-0.18,
        xanchor="center",
        x=0.5,
    ),
)


fig.show()

# Análise da Anomalia em 1991 a 2026
Analise do acúmulo de energia térmica dos oceanos, o aumento da anomalia quente e tendência de aquenciemnto global dos comparada com a média de 1991 a 2020

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import xarray as xr

# Carregar e preparar o dataset
ds = xr.open_dataset("sst_data.nc")
sst = ds["sst"].squeeze()

# Ajuste de coordenadas de lon (0-360 para -180 a 180)
if sst.lon.max() > 180:
    sst = sst.assign_coords(lon=(((sst.lon + 180) % 360) - 180)).sortby("lon")

# Seleção da região do Pacífico Tropical (Niño 3.4: 5°N a 5°S, 170°W a 120°W)
sst_region = sst.sel(lat=slice(5, -5), lon=slice(-170, -120))

# Cálculo da Climatologia de Referência
try:
    climatology_grid = (
        sst_region.sel(time=slice("1961-01-01", "1990-12-31"))
        .groupby("time.month")
        .mean("time")
    )
except KeyError:
    climatology_grid = (
        sst_region.sel(time=slice("1991-01-01", "2020-12-31"))
        .groupby("time.month")
        .mean("time")
    )

ssta_grid = sst_region.groupby("time.month") - climatology_grid

# Ponderação por Cosseno da Latitude e Média Espacial
weights = np.cos(np.deg2rad(ssta_grid.lat))
ssta_series = ssta_grid.weighted(weights).mean(dim=["lat", "lon"]).to_series()

# Recortar o período de interesse (1991 até ao presente)
ssta_series = ssta_series.loc["1991-01-01":]

# Estruturação do DataFrame
df = pd.DataFrame({"anomalia": ssta_series})
df["time"] = df.index

# Média Móvel de 12 Meses
df["tendencia_12m"] = df["anomalia"].rolling(window=12, center=True, min_periods=6).mean()

# Razão de Assimetria: Média Móvel Exponencial (EMA)
df["ema"] = df["anomalia"].ewm(span=36).mean()

# Recortes positivos e negativos para preenchimento de cor
df["pos"] = df["anomalia"].clip(lower=0)
df["neg"] = df["anomalia"].clip(upper=0)

# ARREDONDAMENTO: Garante 2 casas decimais para os dados exibidos no primeiro gráfico
df["anomalia"] = df["anomalia"].round(2)
df["tendencia_12m"] = df["tendencia_12m"].round(2)

# --- CONSTRUÇÃO DO GRÁFICO DUPLO ---
fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.10,
    row_heights=[0.65, 0.35],
    subplot_titles=(
        "Série Temporal de Anomalias de TSM (°C)",
        "Assimetria Térmica Acumulada",
    ),
)

# --- PAINEL 1 ---
# Área Vermelha (Anomalia Quente) - Hover ignorado
fig.add_trace(
    go.Scatter(
        x=df["time"], y=df["pos"], mode="lines", line=dict(width=0),
        fill="tozeroy", fillcolor="rgba(239, 85, 59, 0.45)", name="Anomalia Quente", hoverinfo="skip"
    ), row=1, col=1
)

# Área Azul (Anomalia Fria) - Hover ignorado
fig.add_trace(
    go.Scatter(
        x=df["time"], y=df["neg"], mode="lines", line=dict(width=0),
        fill="tozeroy", fillcolor="rgba(99, 110, 250, 0.45)", name="Anomalia Fria", hoverinfo="skip"
    ), row=1, col=1
)

# Anomalia Mensal Bruta
fig.add_trace(
    go.Scatter(
        x=df["time"], y=df["anomalia"], mode="lines",
        line=dict(color="rgba(200, 200, 200, 0.3)", width=0.5),
        name="Mensal Bruta", hovertemplate="%{x|%b %Y}: <b>%{y:+.2f} °C</b><extra></extra>"
    ), row=1, col=1
)

# Média Móvel (12m)
fig.add_trace(
    go.Scatter(
        x=df["time"], y=df["tendencia_12m"], mode="lines",
        line=dict(color="#FFD700", width=2.2),
        name="Média Móvel (12m)", hovertemplate="Média 12m: <b>%{y:+.2f} °C</b><extra></extra>"
    ), row=1, col=1
)

# --- PAINEL 2: Comparativos da Energia Acumulada Quente vs Fria ---
df["cum_pos"] = df["pos"].cumsum()
df["cum_neg"] = df["neg"].abs().cumsum()

fig.add_trace(
    go.Scatter(
        x=df["time"], y=df["cum_pos"], mode="lines",
        line=dict(color="#EF553B", width=2.5),
        name="Energia Quente Acumulada", hovertemplate="Total Quente: <b>+%{y:.1f} °C·mês</b><extra></extra>"
    ), row=2, col=1
)

fig.add_trace(
    go.Scatter(
        x=df["time"], y=df["cum_neg"], mode="lines",
        line=dict(color="#636EFA", width=2.5),
        name="Energia Fria Acumulada", hovertemplate="Total Frio: <b>-%{y:.1f} °C·mês</b><extra></extra>"
    ), row=2, col=1
)

# --- CONFIGURAÇÃO DO LAYOUT ---
fig.update_layout(
    title=dict(
        text="<b>Análise de Tendência e Assimetria de TSM (1991–Presente)</b>",
        y=0.96,
        x=0.05,
        xanchor="left",
    ),
    template="plotly_dark",
    hovermode="x unified",
    height=750,
    margin=dict(r=30, t=120, l=50, b=50),
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="left",
        x=0.0,
        font=dict(size=11),
    ),
)

fig.add_hline(y=0, line_dash="dash", line_color="white", opacity=0.4, row=1, col=1)

# Ajuste dos eixos Y e X
fig.update_yaxes(title_text="Anomalia (°C)", gridcolor="rgba(255, 255, 255, 0.08)", hoverformat="+.2f", row=1, col=1)
fig.update_yaxes(title_text="Massa Térmica (°C·mês)", gridcolor="rgba(255, 255, 255, 0.08)", row=2, col=1)
fig.update_xaxes(title_text="Ano", gridcolor="rgba(255, 255, 255, 0.08)", row=2, col=1)

fig.show()

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from scipy.stats import linregress
import xarray as xr

# Carregar dataset completo
ds = xr.open_dataset("sst_data.nc")
sst = ds["sst"].squeeze()

if sst.lon.max() > 180:
    sst = sst.assign_coords(lon=(((sst.lon + 180) % 360) - 180)).sortby("lon")

#Definir Linha de Base Climatológica (1991-2020)
climatology_base = (
    sst.sel(time=slice("1991-01-01", "2020-12-31")).groupby("time.month").mean("time")
)

# Anomalia em relação à base 1991-2020 fatiada estritamente até 2025
ssta_grid = (sst.groupby("time.month") - climatology_base).sel(
    time=slice("1991-01-01", "2025-12-31")
)

# Média Global Ponderada pela Latitude
weights = np.cos(np.deg2rad(ssta_grid.lat))
ssta_series = ssta_grid.weighted(weights).mean(dim=["lat", "lon"]).to_series()

# Estruturar DataFrame
df = pd.DataFrame({"anomalia": ssta_series})
df["time"] = df.index
df["tendencia_suave"] = (
    df["anomalia"].rolling(window=60, center=True, min_periods=24).mean()
)

# REGRESSÃO LINEAR (Período 1991–2025)
x_years = (df["time"] - df["time"].iloc[0]).dt.days / 365.25
slope, intercept, r_value, p_value, std_err = linregress(x_years, df["anomalia"])

heating_rate_per_decade = slope * 10
df["reta_tendencia"] = intercept + slope * x_years

# Ajuste visual para preencher abaixo da curva sem gerar componente frio
min_anomaly = df["anomalia"].min()
df["baseline_fill"] = min_anomaly

# Plotar o Gráfico
fig = go.Figure()

# Linha invisível no limite inferior para ancorar o preenchimento vermelho
fig.add_trace(
    go.Scatter(
        x=df["time"],
        y=df["baseline_fill"],
        mode="lines",
        line=dict(width=0),
        showlegend=False,
        hoverinfo="skip",
    )
)

# Preenchimento contínuo de aquecimento (Anomalia Quente)
fig.add_trace(
    go.Scatter(
        x=df["time"],
        y=df["anomalia"],
        mode="lines",
        line=dict(width=0),
        fill="tonexty",
        fillcolor="rgba(239, 85, 59, 0.35)",
        name="Anomalia de TSM",
        hoverinfo="skip",
    )
)

# Sinal Mensal Bruto
fig.add_trace(
    go.Scatter(
        x=df["time"],
        y=df["anomalia"],
        mode="lines",
        line=dict(color="#888888", width=0.7),
        name="Anomalia Mensal",
        hovertemplate="%{x|%b %Y}: %{y:+.2f} °C<extra></extra>",
    )
)

# Média Móvel Suavizada (5 Anos)
fig.add_trace(
    go.Scatter(
        x=df["time"],
        y=df["tendencia_suave"],
        mode="lines",
        line=dict(color="#FFB000", width=2.5),
        name="Média Móvel (5 Anos)",
        hovertemplate="Média 5A: %{y:+.2f} °C<extra></extra>",
    )
)

# Reta de Tendência Linear
fig.add_trace(
    go.Scatter(
        x=df["time"],
        y=df["reta_tendencia"],
        mode="lines",
        line=dict(color="#FF0044", width=3.5, dash="solid"),
        name=f"Tendência Linear (+{heating_rate_per_decade:.2f}°C/década)",
        hovertemplate="Tendência Linear: %{y:+.2f} °C<extra></extra>",
    )
)

# Layout e Estilização
fig.update_layout(
    title=f"<b>Tendência de Aquecimento Global dos Oceanos (1991–2025)</b><br>"
    f"<sup>Taxa de Aquecimento Estimada: <b>+{heating_rate_per_decade:.2f} °C por Década</b> (Base Climatológica 1991–2020)</sup>",
    xaxis_title="Ano",
    yaxis_title="Anomalia de TSM (°C vs Base 1991-2020)",
    yaxis=dict(
        hoverformat="+.2f"
    ),
    template="plotly_dark",
    hovermode="x unified",
    margin=dict(r=20, t=90, l=20, b=80),
    legend=dict(
        orientation="h",
        yanchor="top",
        y=-0.2,
        xanchor="center",
        x=0.5,
    ),
)

fig.add_hline(
    y=0,
    line_dash="dash",
    line_color="white",
    opacity=0.5,
    annotation_text="Média Climatológica (1991-2020)",
)

fig.show()

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from scipy.stats import linregress
import xarray as xr

# Carregar dataset
ds = xr.open_dataset("sst_data.nc")
sst = ds["sst"].squeeze()

if sst.lon.max() > 180:
    sst = sst.assign_coords(lon=(((sst.lon + 180) % 360) - 180)).sortby("lon")

# Definir Linha de Base Climatológica (1991-2020)
climatology_base = (
    sst.sel(time=slice("1991-01-01", "2020-12-31")).groupby("time.month").mean("time")
)

# Anomalia em relação à base 1991-2020 (1991-2026)
ssta_grid = (sst.groupby("time.month") - climatology_base).sel(
    time=slice("1991-01-01", "2026-12-31")
)

# Média Global Ponderada pela Latitude
weights = np.cos(np.deg2rad(ssta_grid.lat))
ssta_series = ssta_grid.weighted(weights).mean(dim=["lat", "lon"]).to_series()

# Estruturar DataFrame e Isolar Componente Frio (<= 0 °C)
df = pd.DataFrame({"anomalia": ssta_series})
df["time"] = df.index
df["neg"] = df["anomalia"].clip(upper=0)

# Média Móvel de 5 Anos na Componente Fria
df["tendencia_suave_fria"] = (
    df["neg"].rolling(window=60, center=True, min_periods=24).mean()
)

# REGRESSÃO LINEAR DA COMPONENTE FRIA (Atenuação/Decaimento do Fenômeno)
x_years = (df["time"] - df["time"].iloc[0]).dt.days / 365.25
slope_neg, intercept_neg, r_value, p_value, std_err = linregress(
    x_years, df["neg"]
)

attenuation_rate_per_decade = slope_neg * 10
df["reta_tendencia_fria"] = intercept_neg + slope_neg * x_years

fig = go.Figure()

# Preenchimento de Área da Anomalia Fria
fig.add_trace(
    go.Scatter(
        x=df["time"],
        y=df["neg"],
        mode="lines",
        line=dict(width=0),
        fill="tozeroy",
        fillcolor="rgba(0, 210, 255, 0.35)",
        name="Anomalia Fria (< 0 °C)",
        hoverinfo="skip",
    )
)

# Sinal Mensal Retificado Frio
fig.add_trace(
    go.Scatter(
        x=df["time"],
        y=df["neg"],
        mode="lines",
        line=dict(color="#00D2FF", width=0.7),
        name="Anomalia Mensal Fria",
        hovertemplate="%{x|%b %Y}: %{y:+.2f} °C<extra></extra>",
    )
)

# Média Móvel Suavizada (5 Anos)
fig.add_trace(
    go.Scatter(
        x=df["time"],
        y=df["tendencia_suave_fria"],
        mode="lines",
        line=dict(color="#FFD700", width=2.5),
        name="Média Móvel (5 Anos)",
        hovertemplate="Média 5A: %{y:+.2f} °C<extra></extra>",
    )
)

# Reta de Tendência Linear (Aproximação do Zero)
fig.add_trace(
    go.Scatter(
        x=df["time"],
        y=df["reta_tendencia_fria"],
        mode="lines",
        line=dict(color="#0055FF", width=3.5, dash="solid"),
        name=f"Atenuação Fria (+{attenuation_rate_per_decade:.2f}°C/década)",
        hovertemplate="Tendência Fria: %{y:+.2f} °C<extra></extra>",
    )
)

# Layout e Estilização
fig.update_layout(
    title=f"<b>Atenuação das Anomalias Frias nos Oceanos (1991–2026)</b><br>"
    f"<sup>Taxa de Retração do Sinal Frio: <b>+{attenuation_rate_per_decade:.2f} °C por Década</b> (Base Climatológica 1991–2020)</sup>",
    xaxis_title="Ano",
    yaxis_title="Anomalia Fria de TSM (°C vs Base 1991-2020)",
    template="plotly_dark",
    hovermode="x unified",
    margin=dict(r=20, t=90, l=20, b=80),
    legend=dict(
        orientation="h",
        yanchor="top",
        y=-0.2,
        xanchor="center",
        x=0.5,
    ),
)

fig.add_hline(
    y=0,
    line_dash="dash",
    line_color="white",
    opacity=0.5,
    annotation_text="Média Climatológica (1991-2020)",
)

fig.show()

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import linregress
import xarray as xr

# Carregar dataset
ds = xr.open_dataset("sst_data.nc")
sst = ds["sst"].squeeze()

if sst.lon.max() > 180:
    sst = sst.assign_coords(lon=(((sst.lon + 180) % 360) - 180)).sortby("lon")

# Climatologia de referência (1991-2020)
climatology_base = (
    sst.sel(time=slice("1991-01-01", "2020-12-31")).groupby("time.month").mean("time")
)

# Anomalias no período (1991–2026)
ssta_grid = (sst.groupby("time.month") - climatology_base).sel(
    time=slice("1991-01-01", "2026-12-31")
)

# Média Ponderada Global
weights = np.cos(np.deg2rad(ssta_grid.lat))
ssta_series = ssta_grid.weighted(weights).mean(dim=["lat", "lon"]).to_series()

df = pd.DataFrame({"anomalia": ssta_series})
df["time"] = df.index
x_years = (df["time"] - df["time"].iloc[0]).dt.days / 365.25

# Componentes truncadas apenas para suporte visual de preenchimento
df["pos"] = df["anomalia"].clip(lower=0)
df["neg"] = df["anomalia"].clip(upper=0)

# Regressão do Sinal Completo (Taxa Real de Aquecimento = ~ +0.16 °C/década)
slope_full, intercept_full, _, _, _ = linregress(x_years, df["anomalia"])
df["reta_full"] = intercept_full + slope_full * x_years

# Regressão da Componente Fria (Taxa de Atenuação/Decaimento)
slope_neg, intercept_neg, _, _, _ = linregress(x_years, df["neg"])
df["reta_neg"] = intercept_neg + slope_neg * x_years

# Médias Móveis (3 Anos = 36 meses)
df["roll_full"] = df["anomalia"].rolling(window=36, center=True, min_periods=12).mean()
df["roll_neg"] = df["neg"].rolling(window=36, center=True, min_periods=12).mean()

# Criar Subplots
fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.08,
    subplot_titles=(
        f"<b>Sinal Global de Aquecimento (°C)</b> (Tendência Real: +{slope_full*10:.2f}°C/década)",
        f"<b>Atenuação das Anomalias Frias (°C)</b> (Decaimento: +{slope_neg*10:.2f}°C/década)",
    ),
)

# --- PAINEL SUPERIOR: SINAL REAL DE AQUECIMENTO E DESTAQUE QUENTE ---
fig.add_trace(
    go.Scatter(
        x=df["time"],
        y=df["pos"],
        mode="lines",
        line=dict(width=0),
        fill="tozeroy",
        fillcolor="rgba(255, 75, 75, 0.35)",
        name="Anomalia Quente (> 0 °C)",
        hoverinfo="skip",
    ),
    row=1,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=df["time"],
        y=df["anomalia"],
        mode="lines",
        line=dict(color="#FF4B4B", width=0.8),
        name="Anomalia Mensal Total",
        hovertemplate="%{x|%b %Y}: %{y:+.2f} °C<extra></extra>",
    ),
    row=1,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=df["time"],
        y=df["roll_full"],
        mode="lines",
        line=dict(color="#FFD700", width=2),
        name="Média Móvel (3 Anos)",
        hovertemplate="Média 3A: %{y:+.2f} °C<extra></extra>",
    ),
    row=1,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=df["time"],
        y=df["reta_full"],
        mode="lines",
        line=dict(color="#FF0000", width=3, dash="solid"),
        name=f"Tendência Linear Total (+{slope_full*10:.2f}°C/década)",
        hovertemplate="Tendência Real: %{y:+.2f} °C<extra></extra>",
    ),
    row=1,
    col=1,
)

# --- PAINEL INFERIOR: ANOMALIA FRIA ---
fig.add_trace(
    go.Scatter(
        x=df["time"],
        y=df["neg"],
        mode="lines",
        line=dict(color="#00D2FF", width=0.8),
        fill="tozeroy",
        fillcolor="rgba(0, 210, 255, 0.3)",
        name="Anomalia Fria (< 0 °C)",
        hovertemplate="%{x|%b %Y}: %{y:+.2f} °C<extra></extra>",
    ),
    row=2,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=df["time"],
        y=df["roll_neg"],
        mode="lines",
        line=dict(color="#FFD700", width=2),
        name="Média Móvel (3 Anos)",
        showlegend=False,
        hovertemplate="Média 3A: %{y:+.2f} °C<extra></extra>",
    ),
    row=2,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=df["time"],
        y=df["reta_neg"],
        mode="lines",
        line=dict(color="#0055FF", width=3, dash="solid"),
        name=f"Tendência Fria (+{slope_neg*10:.2f}°C/década)",
        hovertemplate="Tendência Fria: %{y:+.2f} °C<extra></extra>",
    ),
    row=2,
    col=1,
)

# Layout e Estilização
fig.update_layout(
    height=750,
    template="plotly_dark",
    title="<b>Análise Assimétrica de Anomalias Térmicas (1991–2026)</b>",
    hovermode="x unified",
    margin=dict(r=20, t=80, l=20, b=80),
    legend=dict(
        orientation="h",
        yanchor="top",
        y=-0.12,
        xanchor="center",
        x=0.5,
    ),
)

fig.update_yaxes(title_text="Anomalia (°C)", row=1, col=1)
fig.update_yaxes(title_text="Anomalia (°C)", row=2, col=1)
fig.update_xaxes(title_text="Ano", row=2, col=1)

fig.add_hline(y=0, line_dash="dash", line_color="white", opacity=0.4, row=1, col=1)
fig.add_hline(y=0, line_dash="dash", line_color="white", opacity=0.4, row=2, col=1)

fig.show()

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from scipy.stats import linregress
import xarray as xr

#Carregar dataset sem fatiar prematuramente
ds = xr.open_dataset("sst_data.nc")
sst = ds["sst"].squeeze()

if sst.lon.max() > 180:
    sst = sst.assign_coords(lon=(((sst.lon + 180) % 360) - 180)).sortby("lon")

# Ponderação espacial pelo cosseno da latitude
weights = np.cos(np.deg2rad(sst.lat))
sst_weighted = sst.weighted(weights).mean(dim=["lat", "lon"])

# Fatiar a série global no período de interesse (1991-2026)
sst_period = sst_weighted.sel(time=slice("1991-01-01", "2026-12-31"))

# Agrupar por ANO para obter a Média Anual Global
sst_annual = sst_period.groupby("time.year").mean("time").to_series()

# Calcular a Média do Período de Referência Climatológica (1991-2020)
base_mean = sst_annual.loc[1991:2020].mean()

# Criar DataFrame com as Anomalias Anuais
df_annual = pd.DataFrame(
    {
        "ano": sst_annual.index,
        "temp_media": sst_annual.values,
        "anomalia": sst_annual.values - base_mean,
    }
)

df_annual["color"] = np.where(
    df_annual["anomalia"] >= 0,
    "rgba(239, 85, 59, 0.85)",
    "rgba(0, 210, 255, 0.85)",
)

# Regressão Linear
slope, intercept, r_value, p_value, std_err = linregress(
    df_annual["ano"], df_annual["anomalia"]
)
df_annual["tendencia"] = intercept + slope * df_annual["ano"]
taxa_decada = slope * 10

# Plotar o Gráfico com Legenda Inferior
fig = go.Figure()

fig.add_trace(
    go.Bar(
        x=df_annual["ano"],
        y=df_annual["anomalia"],
        marker_color=df_annual["color"],
        name="Anomalia Anual (°C)",
        customdata=df_annual["temp_media"],
        hovertemplate="<b>Ano %{x}</b><br>"
        + "Média Absoluta: %{customdata:.2f} °C<br>"
        + "Diferença da Base: %{y:+.3f} °C<extra></extra>",
    )
)

fig.add_trace(
    go.Scatter(
        x=df_annual["ano"],
        y=df_annual["tendencia"],
        mode="lines",
        line=dict(color="#FFD700", width=3),
        name=f"Tendência Linear (+{taxa_decada:.2f}°C/década)",
        hovertemplate="Tendência: %{y:+.3f} °C<extra></extra>",
    )
)

fig.add_hline(
    y=0,
    line_dash="dash",
    line_color="white",
    opacity=0.8,
    annotation_text=f"Média Climatológica 1991–2020 ({base_mean:.2f}°C)",
    annotation_position="bottom left",
)

fig.update_layout(
    title=f"<b>Aumento da Temperatura Média Anual dos Oceanos (1991–2026)</b><br>"
    f"<sup>Comparação com a Média Base 1991–2020 ({base_mean:.2f} °C) — Taxa: <b>+{taxa_decada:.2f} °C/década</b></sup>",
    xaxis=dict(title="Ano", dtick=2),
    yaxis_title="Diferença em relação à Média 1991–2020 (°C)",
    template="plotly_dark",
    hovermode="x unified",
    margin=dict(r=20, t=90, l=20, b=80),
    legend=dict(
        orientation="h",
        yanchor="top",
        y=-0.2,
        xanchor="center",
        x=0.5,
    ),
)

fig.show()

In [ ]:
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from IPython.display import HTML, display  # Para visualização interativa em Notebooks
from matplotlib.animation import FuncAnimation
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import xarray as xr

# =============================================================================
#  Carregamento dos Dados e Cálculo da Climatologia (1991-2020)
# =============================================================================
# Carrega o dataset global de Temperatura da Superfície do Mar (TSM)
ds = xr.open_dataset('sst_data.nc')

# Climatologia Mensal Média de referência (1991-2020)
climatologia = (
    ds['sst']
    .sel(time=slice('1991-01-01', '2020-12-31'))
    .groupby('time.month')
    .mean('time')
)

# =============================================================================
#  Seleção das Séries Temporais dos Eventos (2015–2026)
# =============================================================================
sst_2015 = ds['sst'].sel(time=slice('2015-01-01', '2015-12-31'))
sst_lanina = ds['sst'].sel(time=slice('2022-01-01', '2022-12-31'))
sst_elnino = ds['sst'].sel(time=slice('2023-01-01', '2023-12-31'))
sst_2026 = ds['sst'].sel(time=slice('2026-01-01', '2026-12-31'))

nomes_meses = [
    'Janeiro',
    'Fevereiro',
    'Março',
    'Abril',
    'Maio',
    'Junho',
    'Julho',
    'Agosto',
    'Setembro',
    'Outubro',
    'Novembro',
    'Dezembro',
]

num_meses_2026 = len(sst_2026.time)

# =============================================================================
#  Configuração da Figura, Margens e Barra de Cores Lateral
# =============================================================================
fig, axes = plt.subplots(
    2,
    2,
    figsize=(16, 9),
    subplot_kw={'projection': ccrs.PlateCarree(central_longitude=180)},
    facecolor='#0e0e0e',
)
(ax1, ax2), (ax3, ax4) = axes

# Reserva espaço ao redor dos mapas:
# top=0.88 abre espaço para o título superior | right=0.88 abre espaço para a barra de cores
fig.subplots_adjust(
    top=0.88, bottom=0.08, left=0.04, right=0.88, hspace=0.22, wspace=0.10
)

# Criamos uma escala de cores normalizada para a barra de cores unificada
norma_cor = mcolors.Normalize(vmin=-3.0, vmax=3.0)
mapeador_cor = cm.ScalarMappable(cmap='RdBu_r', norm=norma_cor)
mapeador_cor.set_array([])

# Adicionamos um eixo dedicado para a barra de cores: [esquerda, baixo, largura, altura]
eixo_cbar = fig.add_axes([0.90, 0.12, 0.018, 0.72])
cbar = fig.colorbar(mapeador_cor, cax=eixo_cbar)
cbar.set_label('Anomalia de TSM (°C)', color='white', fontsize=11, labelpad=12)
cbar.ax.yaxis.set_tick_params(color='white', labelcolor='white')


# =============================================================================
#  Função de Atualização da Animação (Frame a Frame)
# =============================================================================
def atualizar_frame(frame):
  # Limpa o conteúdo dos eixos a cada frame
  ax1.clear()
  ax2.clear()
  ax3.clear()
  ax4.clear()

  mes_atual_idx = frame + 1
  clima_m = climatologia.sel(month=mes_atual_idx)

  # --- Cálculo das Anomalias de Temperatura (Δ °C) ---
  anomalia_elnino = sst_elnino.isel(time=frame) - clima_m
  anomalia_lanina = sst_lanina.isel(time=frame) - clima_m
  anomalia_2015 = sst_2015.isel(time=frame) - clima_m

  # Lógica de congelamento para 2026 caso os dados do ano estejam incompletos
  idx_26 = min(frame, num_meses_2026 - 1)
  sst_26_m = sst_2026.isel(time=idx_26)
  mes_real_2026 = int(sst_26_m.time.dt.month)
  clima_congelado_2026 = climatologia.sel(month=mes_real_2026)
  anomalia_2026 = sst_26_m - clima_congelado_2026

  # --- Configuração Geográfica dos Mapas ---
  for ax in [ax1, ax2, ax3, ax4]:
    ax.set_global()
    ax.add_feature(cfeature.COASTLINE, linewidth=0.6, edgecolor='#888888')
    ax.add_feature(cfeature.LAND, facecolor='#1a1a1a')
    ax.set_facecolor('#0e0e0e')

  # --- Renderização dos Painéis ---
  # Painel 1 (Topo Esquerda)
  ax1.pcolormesh(
      anomalia_elnino.lon,
      anomalia_elnino.lat,
      anomalia_elnino.values,
      transform=ccrs.PlateCarree(),
      cmap='RdBu_r',
      vmin=-3.0,
      vmax=3.0,
      shading='gouraud',
  )
  ax1.set_title(
      f'Anomalia Último El Niño - 2023 ({nomes_meses[frame]})',
      color='white',
      fontsize=11,
      fontweight='bold',
      pad=8,
  )

  # Painel 2 (Topo Direita)
  ax2.pcolormesh(
      anomalia_lanina.lon,
      anomalia_lanina.lat,
      anomalia_lanina.values,
      transform=ccrs.PlateCarree(),
      cmap='RdBu_r',
      vmin=-3.0,
      vmax=3.0,
      shading='gouraud',
  )
  ax2.set_title(
      f'Anomalia Última La Niña - 2022 ({nomes_meses[frame]})',
      color='white',
      fontsize=11,
      fontweight='bold',
      pad=8,
  )

  # Painel 3 (Baixo Esquerda)
  ax3.pcolormesh(
      anomalia_2015.lon,
      anomalia_2015.lat,
      anomalia_2015.values,
      transform=ccrs.PlateCarree(),
      cmap='RdBu_r',
      vmin=-3.0,
      vmax=3.0,
      shading='gouraud',
  )
  ax3.set_title(
      f'Anomalia El Niño Histórico - 2015 ({nomes_meses[frame]})',
      color='white',
      fontsize=11,
      fontweight='bold',
      pad=8,
  )

  # Painel 4 (Baixo Direita)
  titulo_26 = f'Anomalia 2026 ({nomes_meses[mes_real_2026-1]})' + (
      ' [Atualizando...]' if frame >= num_meses_2026 else ''
  )
  ax4.pcolormesh(
      anomalia_2026.lon,
      anomalia_2026.lat,
      anomalia_2026.values,
      transform=ccrs.PlateCarree(),
      cmap='RdBu_r',
      vmin=-3.0,
      vmax=3.0,
      shading='gouraud',
  )
  ax4.set_title(
      titulo_26, color='white', fontsize=11, fontweight='bold', pad=8
  )

  # Título Principal Elevado (y=0.95) para não colidir com os subplots
  fig.suptitle(
      f'Comparativo de Anomalias de TSM Global (Δ °C) — Mês:'
      f' {nomes_meses[frame]}',
      color='white',
      fontsize=15,
      fontweight='bold',
      y=0.95,
  )


# =============================================================================
# Execução da Animação
# =============================================================================
anim = FuncAnimation(fig, atualizar_frame, frames=12, interval=800, repeat=False)

# Para reprodução no Jupyter Notebook / Google Colab:
plt.close()
display(HTML(anim.to_jshtml()))

# Para uso em scripts Python comuns (VS Code / PyCharm / Terminal):
# plt.show()

In [ ]:
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib.animation import FuncAnimation, PillowWriter
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import xarray as xr

# =============================================================================
# Carregamento e Preparação dos Dados NetCDF
# =============================================================================
# Carrega o conjunto de dados global de TSM
ds = xr.open_dataset('sst_data.nc')

# Climatologia Mensal Média de referência (1991-2020)
climatologia = (
    ds['sst']
    .sel(time=slice('1991-01-01', '2020-12-31'))
    .groupby('time.month')
    .mean('time')
)

# Seleção das séries temporais para os diferentes anos/eventos
sst_2015 = ds['sst'].sel(time=slice('2015-01-01', '2015-12-31'))
sst_2022 = ds['sst'].sel(time=slice('2022-01-01', '2022-12-31'))
sst_2023 = ds['sst'].sel(time=slice('2023-01-01', '2023-12-31'))
sst_2026 = ds['sst'].sel(time=slice('2026-01-01', '2026-12-31'))

nomes_meses = [
    'Janeiro',
    'Fevereiro',
    'Março',
    'Abril',
    'Maio',
    'Junho',
    'Julho',
    'Agosto',
    'Setembro',
    'Outubro',
    'Novembro',
    'Dezembro',
]

# Quantidade de meses disponíveis em 2026
num_meses_2026 = len(sst_2026.time)

# =============================================================================
# Configuração da Figura, Margens e Barra de Cores Lateral
# =============================================================================
fig, axes = plt.subplots(
    2,
    2,
    figsize=(16, 9),
    subplot_kw={'projection': ccrs.PlateCarree(central_longitude=180)},
    facecolor='#0e0e0e',
)
(ax1, ax2), (ax3, ax4) = axes

fig.subplots_adjust(
    top=0.88, bottom=0.08, left=0.04, right=0.88, hspace=0.22, wspace=0.10
)

# Criação da barra de cores unificada
norma_cor = mcolors.Normalize(vmin=-3.0, vmax=3.0)
mapeador_cor = cm.ScalarMappable(cmap='RdBu_r', norm=norma_cor)
mapeador_cor.set_array([])

# Adiciona o eixo da barra de cores
eixo_cbar = fig.add_axes([0.90, 0.12, 0.018, 0.72])
cbar = fig.colorbar(mapeador_cor, cax=eixo_cbar)
cbar.set_label(
    'Anomalia de TSM (Δ °C)', color='white', fontsize=11, labelpad=12
)
cbar.ax.yaxis.set_tick_params(color='white', labelcolor='white')


# =============================================================================
# Função de Atualização Frame a Frame (12 Meses)
# =============================================================================
def atualizar_frame(frame):
  # Limpa o conteúdo dos subplots a cada frame
  ax1.clear()
  ax2.clear()
  ax3.clear()
  ax4.clear()

  mes_atual_idx = frame + 1  # Mês de 1 a 12
  clima_m = climatologia.sel(month=mes_atual_idx)

  # Cálculo das Anomalias para anos completos (2015, 2022, 2023)
  anomalia_15_m = sst_2015.isel(time=frame) - clima_m
  anomalia_22_m = sst_2022.isel(time=frame) - clima_m
  anomalia_23_m = sst_2023.isel(time=frame) - clima_m

  # Lógica de Congelamento para o ano de 2026 (caso incompleto)
  idx_26 = min(frame, num_meses_2026 - 1)
  sst_26_m = sst_2026.isel(time=idx_26)
  mes_real_2026 = int(sst_26_m.time.dt.month)
  clima_congelado_2026 = climatologia.sel(month=mes_real_2026)
  anomalia_26_m = sst_26_m - clima_congelado_2026

  # Configuração Geográfica Comum para todos os painéis
  for ax in [ax1, ax2, ax3, ax4]:
    ax.set_global()
    ax.add_feature(cfeature.COASTLINE, linewidth=0.6, edgecolor='#888888')
    ax.add_feature(cfeature.LAND, facecolor='#1a1a1a')
    ax.set_facecolor('#0e0e0e')

  # --- Painel 1: Anomalia El Niño 2023 ---
  ax1.pcolormesh(
      anomalia_23_m.lon,
      anomalia_23_m.lat,
      anomalia_23_m.values,
      transform=ccrs.PlateCarree(),
      cmap='RdBu_r',
      vmin=-3.0,
      vmax=3.0,
      shading='gouraud',
  )
  ax1.set_title(
      f'Anomalia El Niño - 2023 ({nomes_meses[frame]})',
      color='white',
      fontsize=10,
      fontweight='bold',
      pad=8,
  )

  # --- Painel 2: Anomalia La Niña 2022  ---
  ax2.pcolormesh(
      anomalia_22_m.lon,
      anomalia_22_m.lat,
      anomalia_22_m.values,
      transform=ccrs.PlateCarree(),
      cmap='RdBu_r',
      vmin=-3.0,
      vmax=3.0,
      shading='gouraud',
  )
  ax2.set_title(
      f'Anomalia La Niña - 2022 ({nomes_meses[frame]})',
      color='white',
      fontsize=10,
      fontweight='bold',
      pad=8,
  )

  # --- Painel 3: Anomalia El Niño Histórico 2015 ---
  ax3.pcolormesh(
      anomalia_15_m.lon,
      anomalia_15_m.lat,
      anomalia_15_m.values,
      transform=ccrs.PlateCarree(),
      cmap='RdBu_r',
      vmin=-3.0,
      vmax=3.0,
      shading='gouraud',
  )
  ax3.set_title(
      f'Anomalia El Niño Histórico - 2015 ({nomes_meses[frame]})',
      color='white',
      fontsize=10,
      fontweight='bold',
      pad=8,
  )

  # --- Painel 4: Anomalia 2026 vs Climatologia ---
  titulo_26 = f'Anomalia 2026 ({nomes_meses[mes_real_2026-1]})' + (
      ' [Atualizando...]' if frame >= num_meses_2026 else ''
  )
  ax4.pcolormesh(
      anomalia_26_m.lon,
      anomalia_26_m.lat,
      anomalia_26_m.values,
      transform=ccrs.PlateCarree(),
      cmap='RdBu_r',
      vmin=-3.0,
      vmax=3.0,
      shading='gouraud',
  )
  ax4.set_title(
      titulo_26, color='white', fontsize=10, fontweight='bold', pad=8
  )

  # Título Principal bem posicionado no topo (y=0.96)
  fig.suptitle(
      f'Evolução Global das Anomalias de TSM — Mês: {nomes_meses[frame]}',
      color='white',
      fontsize=15,
      fontweight='bold',
      y=0.96,
  )


# =============================================================================
# Gerar e Salvar o GIF Animado
# =============================================================================
print('Gerando a animação GIF...')

# Cria a animação para os 12 meses
anim = FuncAnimation(fig, atualizar_frame, frames=12, interval=800)

# Configura o gravador PillowWriter (1.25 quadros por segundo)
writer = PillowWriter(fps=1.25)
anim.save('evolucao_global_tsm_2026.gif', writer=writer)
plt.close()

print(
    'GIF gerado com sucesso e salvo como'
    " 'evolucao_global_tsm_2026.gif'."
)